In [1]:
import os
import json
import datasets

sft_data_dir = '/home/jadeleiyu/projects/mbrl_agent/world_model/sft_data'
n_jobs = 8

wm_cot_data = []
for job_id in range(n_jobs):
    try:
        with open(os.path.join(sft_data_dir, f"wm_cot_gen_outputs_{job_id}.json"), 'r') as f:
            wm_cot_data_i = json.load(f)
        wm_cot_data += wm_cot_data_i
    except FileNotFoundError as e:
        pass


/home/jadeleiyu/miniforge3/envs/mbrl_agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!nvidia-smi

Wed Sep 24 13:56:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:53:00.0 Off |                    0 |
| N/A   36C    P0             97W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

WM_SYS_PROMPT = """You are an intelligent agent that predicts next state from given current action in a web environment, with your own logical reasoning. 

Here's the information you'll have:
The user's objective: This is the task you're trying to complete.
The current web page's accessibility tree: This is a simplified representation of the webpage, providing key information.
The current web page's URL: This is the page you're currently navigating.
The previous action: This is the action you just performed in the previous step. It may be helpful to track your progress. 
The current action: This is the current action that you performed to achieve the user's objective in the current web page's accessibility tree.
The format of previous actions can fall into several categories:
Page Operation Actions:

```click [id]```: This action clicks on an element with a specific id on the webpage.
```type [id] [content]```: Use this to type the content into the field with id. By default, the 'Enter' key is pressed after typing unless press_enter_after is set to 0, i.e., ```type [id] [content] [0]```.
```hover [id]```: Hover over an element with id.
```press [key_comb]```: Simulates the pressing of a key combination on the keyboard (e.g., Ctrl+v).
```scroll [down]``` or ```scroll [up]```: Scroll the page up or down.

Tab Management Actions:
```new_tab```: Open a new, empty browser tab.
```tab_focus [tab_index]```: Switch the browser's focus to a specific tab using its index.
```close_tab```: Close the currently active tab.

URL Navigation Actions:
```goto [url]```: Navigate to a specific URL.
```go_back```: Navigate to the previously viewed page.
```go_forward```: Navigate to the next page (if a previous 'go_back' action was performed)

Completion Action:
```stop [answer]```: Done when you believe the task is complete.

Given the information above, you should first perform reasoning to predict expected changes on the current web page's accessibility tree,
and then generate the resulting next web page's accessibility tree based on your predicted web page changes.
Generate your answer in the following format: 
[Web state changes]\n\{changes\}\n\n\n[Next page accessibility tree]\n\{next_acc_tree\}
where ``changes`` are the predicted web page changes, and ``next_acc_tree`` is your predicted next page accessibility tree.
"""

WM_USR_PROMPT_TEMPLATE = """User objective: {usr_obj}
Current web page accessibility tree: {curr_acc_tree}
Current web page URL: {curr_url}
Previous action: {prev_action}
Current action: {curr_action}
"""

answer_parse_pattern_cot = "[Web state changes]"
answer_parse_pattern_obs = "[Next page accessibility tree]"

def parse_output(response):
    try:
        delta_and_obs = response.split(answer_parse_pattern_cot)[-1]
        delta, obs = delta_and_obs.split(answer_parse_pattern_obs)
        return delta, obs
    except Exception as e:
        return "None", "None"

INFO 09-24 13:56:55 [__init__.py:241] Automatically detected platform cuda.


In [5]:
# wm_model_name="unsloth/gpt-oss-120b-BF16"

wm_model_name="/checkpoint/multimodal-reasoning/jadeleiyu/mbrl_agent/wm_sft/gpt-oss-120b-llamafactory-merged-100"

world_model = LLM(
    model=wm_model_name, 
    trust_remote_code=True,
    tensor_parallel_size=8,
    dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained(wm_model_name)

INFO 09-24 13:56:59 [utils.py:326] non-default args: {'model': '/checkpoint/multimodal-reasoning/jadeleiyu/mbrl_agent/wm_sft/gpt-oss-120b-llamafactory-merged-100', 'trust_remote_code': True, 'dtype': 'bfloat16', 'tensor_parallel_size': 8, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 09-24 13:57:07 [__init__.py:711] Resolved architecture: GptOssForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 09-24 13:57:07 [__init__.py:1750] Using max model len 131072


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  99%|█████████████████████████████████████▌| 82/83 [00:07<00:00,  5.77it/s]

(EngineCore_0 pid=343894) (VllmWorker TP3 pid=343925) INFO 09-24 14:12:47 [custom_all_reduce.py:196] Registering 6059 cuda graph addresses
(EngineCore_0 pid=343894) (VllmWorker TP4 pid=343928) INFO 09-24 14:12:47 [custom_all_reduce.py:196] Registering 6059 cuda graph addresses
(EngineCore_0 pid=343894) 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████████████████████████████████| 83/83 [00:07<00:00, 10.42it/s]

(VllmWorker TP7 pid=343937) 

INFO 09-24 14:12:47 [custom_all_reduce.py:196] Registering 6059 cuda graph addresses
(EngineCore_0 pid=343894) (VllmWorker TP0 pid=343916) INFO 09-24 14:12:47 [custom_all_reduce.py:196] Registering 6059 cuda graph addresses
(EngineCore_0 pid=343894) (VllmWorker TP2 pid=343921) INFO 09-24 14:12:47 [custom_all_reduce.py:196] Registering 6059 cuda graph addresses
(EngineCore_0 pid=343894) (VllmWorker TP7 pid=343937) INFO 09-24 14:12:48 [gpu_model_runner.py:2708] Graph capturing finished in 9 secs, took 1.04 GiB
(EngineCore_0 pid=343894) (VllmWorker TP3 pid=343925) INFO 09-24 14:12:48 [gpu_model_runner.py:2708] Graph capturing finished in 9 secs, took 1.04 GiB
(EngineCore_0 pid=343894) (VllmWorker TP6 pid=343934) INFO 09-24 14:12:48 [gpu_model_runner.py:2708] Graph capturing finished in 9 secs, took 1.04 GiB
(EngineCore_0 pid=343894) (EngineCore_0 pid=343894) (EngineCore_0 pid=343894) (VllmWorker TP0 pid=343916) (VllmWorker TP2 pid=343921) (VllmWorker TP4 pid=343928) (EngineCore_0 pid=3438

In [6]:
from tqdm import tqdm
import torch

torch.cuda.empty_cache()

sampling_params = SamplingParams(
    max_tokens=8192,
    temperature=1.0,
    top_p=0.9
)
batch_size = 16
n_batch = 16

responses, deltas, next_obs_trees = [], [], []

for i in tqdm(range(n_batch)):
    start, end = i*batch_size, (i+1)*batch_size
    batch_input_prompts = []
    for j in range(start, end):
        example = wm_cot_data[j]
        messages = [
            {'role':'system', 'content': WM_SYS_PROMPT},
            {'role': 'user', 'content': WM_USR_PROMPT_TEMPLATE.format(
                usr_obj=example['objective'],
                curr_acc_tree=example['current_observation'],
                curr_url=example['current_url'],
                prev_action=example['previous_action'],
                curr_action=example['current_action']
            )},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,  # adds the assistant turn to complete
        )
        batch_input_prompts.append(prompt)

    outputs = world_model.generate(
        batch_input_prompts,
        sampling_params=sampling_params,
        use_tqdm=False
    )
    batch_delta, batch_next_obs = [], []
    for out in outputs:
        response = out.outputs[0].text
        delta, next_obs = parse_output(response)
        responses.append(response)
        deltas.append(delta)
        next_obs_trees.append(next_obs)


  0%|                                                                                                        | 0/16 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [16:52<00:00, 63.27s/it]


In [18]:
print(responses[15])


analysisWe have current page (Apple Newsroom) and the user action is "click [1003]". Let's locate element [1003] in the accessibility tree. In the tree, [1003] is a button inside a list item:

From the tree:

```
[1002] listitem ''
    [1003] button 'Apple Intelligence now features Image Playground, Genmoji, and\xa0more - PRESS RELEASE - Posted on December 11, 2024', clickable, visible
        [1005] image '...' 
```

So clicking that button likely navigates to the article page for that news item. Therefore the next page will be the article page at URL: https://www.apple.com/newsroom/2024/12/apple-intelligence-now-features-image-playground-genmoji-and-more/ (maybe). The URL from the link is provided in the button? Actually it's a button, but may act like a link. The link in the list item is a button; but typical Newsroom uses <a> tags, maybe they are implemented as button for click to navigate. So clicking will navigate to the article page.

Thus next page accessibility tree will be a 

In [17]:
deltas[5]

'None'

In [10]:
next_obs_trees[5:15]

['None',
 '**\nRootWebArea \'The 10 Best Las Vegas Hotels (From $64)\', focused, url=\'https://www.booking.com/city/us/las-vegas.html\'\n\t[167] link \'Skip to main content\', clickable, url=\'https://www.booking.com/city/us/las-vegas.html#indexsearch\'\n\t\tStaticText \'Skip to main content\'\n\t[173] banner \'\', visible\n\t\t[174] navigation \'\', visible\n\t\t\t[177] link \'Booking.com Online Hotel Reservations\', clickable, visible, url=\'https://www.booking.com/index.html\'\n\t\t\t[181] button \'Choose your currency. Your current currency is U.S. Dollar\', clickable, visible\n\t\t\t\tStaticText \'Choose your currency. Your current currency is U.S. Dollar\'\n\t\t\t[186] button \'Choose your language. Your current language is English (US)\', clickable, visible\n\t\t\t\tStaticText \'Choose your language. Your current language is English (US)\'\n\t\t\t[193] link \'Get help with your reservation\', clickable, visible, url=\'https://secure.booking.com/help.html\'\n\t\t\t\tStaticText \'

In [19]:
with open("/home/jadeleiyu/projects/mbrl_agent/world_model/sample_outputs/gpt-oss-120b-sft-100-responses.json", 'w') as f:
    json.dump(responses, f)
